In [ ]:
from google.colab import drive
drive.mount('/content/drive')


# Geometry-V1 D0 all-layer discovery handoff

Run once, top to bottom. D0 is representation/equivariance discovery only: `science_denominator=0`; no method, detector, or scientific conclusion follows. The runner creates its fixed procedural RGB references and attacks. Known H stays evaluation truth inside that runner.


In [ ]:
import json, os, pathlib, subprocess, sys
from datetime import datetime, timezone
from google.colab import userdata

REPO_URL = 'https://github.com/RICHAAARC/CEG-WM.git'
BRANCH = 'Geometry-V1'
RUNNER_PATH = 'experiments/run_geometry_v1_qk_all_layer_discovery_operational.py'
SUCCESS_PREFIX = 'CEGWM_GEOMETRY_V1_QK_D0 '
FAILURE_PREFIX = 'CEGWM_GEOMETRY_V1_QK_D0_FAILURE '
MAX_CONTROL_BYTES = 1024
D0_FAILED = False
RUNNER_ATTEMPTED = False

def stop(stage, error):
    global D0_FAILED
    if not D0_FAILED:
        D0_FAILED = True
        print('CEGWM_GEOMETRY_V1_QK_D0_HANDOFF_FAILURE ' + json.dumps({'stage': stage, 'error_class': type(error).__name__}, sort_keys=True, separators=(',', ':')))

def git_output(repo, *args):
    return subprocess.run(['git', *args], cwd=repo, check=True, capture_output=True, text=True).stdout.strip()


In [ ]:
if not D0_FAILED:
    try:
        session_utc = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        repo = pathlib.Path('/content') / ('geometry-v1-qk-d0-' + session_utc)
        if repo.exists(): raise FileExistsError('fresh checkout path exists')
        subprocess.run(['git', 'clone', '--single-branch', '--branch', BRANCH, REPO_URL, str(repo)], check=True)
        execution_commit = git_output(repo, 'rev-parse', 'HEAD')
        checkout_branch = git_output(repo, 'branch', '--show-current')
        checkout_clean = git_output(repo, 'status', '--porcelain') == ''
        if checkout_branch != BRANCH or not checkout_clean: raise RuntimeError('checkout identity mismatch')
        print('CEGWM_GEOMETRY_V1_QK_D0_CHECKOUT ' + json.dumps({'branch': checkout_branch, 'commit': execution_commit, 'clean': checkout_clean}, sort_keys=True))
        subprocess.run([sys.executable, '-m', 'pip', 'install', str(repo)], check=True)
    except BaseException as error:
        stop('checkout_or_install', error)


In [ ]:
hf_token = ''
runner_env = control_read = control_write = None
if not D0_FAILED:
    try:
        if RUNNER_ATTEMPTED: raise RuntimeError('runner already attempted')
        RUNNER_ATTEMPTED = True
        hf_token = userdata.get('HF_TOKEN')
        if not isinstance(hf_token, str) or not hf_token.strip(): raise RuntimeError('HF_TOKEN Colab Secret is required')
        drive_root = pathlib.Path('/content/drive/MyDrive/CEG-WM/Geometry-V1/D0')
        run_dir = drive_root / ('Geometry-V1-QK-D0-' + execution_commit[:12] + '-' + session_utc)
        if run_dir.exists(): raise FileExistsError('create-only Drive run directory exists')
        runner_env = {name: value for name, value in os.environ.items() if 'TOKEN' not in name.upper()}
        runner_env['HF_TOKEN'] = hf_token
        hf_token = ''
        control_read, control_write = os.pipe()
        command = [sys.executable, RUNNER_PATH, '--repo-root', str(repo), '--expected-exact', execution_commit, '--output-root', str(run_dir), '--control-fd', str(control_write)]
        process = subprocess.Popen(command, cwd=repo, env=runner_env, pass_fds=(control_write,), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        os.close(control_write); control_write = None
        runner_rc = process.wait(timeout=7200)
        line = os.read(control_read, MAX_CONTROL_BYTES + 1)
        if len(line) > MAX_CONTROL_BYTES or not line.endswith(b'\n'): raise RuntimeError('invalid compact control receipt')
        text = line.decode('utf-8', 'strict').strip()
        if text.startswith(SUCCESS_PREFIX): receipt_status, receipt = 'success', json.loads(text[len(SUCCESS_PREFIX):])
        elif text.startswith(FAILURE_PREFIX): receipt_status, receipt = 'failure', json.loads(text[len(FAILURE_PREFIX):])
        else: raise RuntimeError('unexpected control prefix')
        if receipt.get('run_id') != 'geometry-v1-qk-d0-' + execution_commit[:12]: raise RuntimeError('receipt identity mismatch')
        terminal = {'run_id': receipt.get('run_id'), 'exact': execution_commit, 'd0_status': receipt.get('d0_status'), 'artifact_status': receipt.get('artifact_status'), 'selected_layer_paths': receipt.get('selected_layer_paths', []), 'science_denominator': 0, 'drive_directory': str(run_dir), 'runner_rc': runner_rc}
        print('CEGWM_GEOMETRY_V1_QK_D0_TERMINAL ' + json.dumps(terminal, sort_keys=True, separators=(',', ':')))
        if receipt_status != 'success': raise RuntimeError('runner reported failure')
    except BaseException as error:
        stop('runner', error)
    finally:
        hf_token = ''
        if runner_env is not None: runner_env.pop('HF_TOKEN', None)
        for fd in (control_read, control_write):
            if fd is not None: os.close(fd)


The notebook displays only the bounded terminal receipt. It never reads ZIP bytes, calculates hashes, retries, switches layers, falls back, tunes, or chooses a route per sample. A failure remains in the create-only Drive directory and stops.
